# Student-selected longer training at half-life four

**13 October 2026.** The student chose **“Test both longer-training variants (Recommended)”** after reviewing the shorter-training result. This executes only the declared reference and two variants. No automatic winner selection, subsequent search or Kaggle upload occurs.

| Recipe | XGBoost trees | LightGBM trees |
|---|---:|---:|
| Public reference (0.95471) | 200 | 150 |
| Longer XGBoost | 300 | 150 |
| Longer LightGBM | 200 | 225 |

Both alternatives change one component's number of trees. Half-life four, learning rate 0.05, 15 leaves, original bin and minimum-leaf settings, lambda 1, seed 42, two CPU threads and the equal probability average remain unchanged. Unknown rows do not receive invented labels or enter supervised fitting.

## Hypothesis and evaluation

Shorter training reduced the checking means for both individual components and the pair. This motivates a bounded test of more corrections under the current recency weighting. It does not prove that longer training will help. The earlier longer-LightGBM test used equal row weights and performed worse; that result remains relevant evidence of overfitting risk.

The selected protocol uses the same 15 fourteen-period checks and later training cutoffs 24, 28 and 31 checking through 35. Each XGBoost time-feature filter is learned from confirmed fitting rows through cutoff minus three. Reference and candidates share each check's mask. Later checks forecast 11, 7 and 4 periods, so they are reported separately and cannot be treated as an expected fourteen-period Kaggle result. All checking periods overlap earlier development data.

Verified 200-tree XGBoost and 150-tree LightGBM component predictions can be reused when their data hashes, parameters, package versions, feature lists, fitting labels and checking IDs match. The original fitting function is unchanged. In a fresh folder with no cache, the notebook fits every required component using the supplied competition files alone.

In [1]:
from pathlib import Path
RUN_TRAINING = True
STUDENT_CHOICE = 'both'
DATA_DIR = Path('/Users/mlvlo/BT4012/Kaggle/Data Set')
OUTPUT_DIR = Path('/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/half4_longer_training_results')


In [2]:
"""Proposed fixed comparison. Default execution checks data without fitting models.

After the student's modeling choice, --run --choice both tests the two declared
longer-training variants. --choice lgb_only tests just the LightGBM change.
There is no automatic candidate selection, follow-on search or Kaggle upload.
"""
from pathlib import Path
import argparse
import hashlib
import json
import time

import numpy as np
import pandas as pd
import sklearn
import xgboost
import lightgbm
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

VERSION = 'fixed-half4-training-length-v1'
FEATURES = [f'feat_{i}' for i in range(1, 166)]
HALF_LIFE = 4
FOLDS = [(k, k + 1, k + 14, 'original_14_period') for k in range(7, 22)]
FOLDS += [(k, k + 1, 35, 'later_cutoff') for k in (24, 28, 31)]
XGB_SETTINGS = dict(learning_rate=.05, tree_method='hist', grow_policy='lossguide',
                    max_depth=0, max_leaves=15, max_bin=127, min_child_weight=1,
                    reg_lambda=1, objective='binary:logistic', n_jobs=2, random_state=42)
LGB_SETTINGS = dict(learning_rate=.05, num_leaves=15, max_bin=63, min_child_samples=5,
                    min_child_weight=.001, reg_lambda=1, objective='binary',
                    deterministic=True, force_col_wise=True, verbosity=-1,
                    n_jobs=2, random_state=42)
RECIPES = {'reference_200_150': (200, 150), 'long_xgb_300_150': (300, 150),
           'long_lgb_200_225': (200, 225)}
EXPECTED_HASHES = {
    'Train Data.csv': '6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706',
    'Test Data.csv': '344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db',
    'Sample Submission/BT4012 Competition Submission.csv':
        '3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e',
}

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_provided_data(data_dir):
    hashes = {name: digest(data_dir / name) for name in EXPECTED_HASHES}
    assert hashes == EXPECTED_HASHES, 'Provided data differs from the reviewed reference.'
    train = pd.read_csv(data_dir / 'Train Data.csv')
    test = pd.read_csv(data_dir / 'Test Data.csv')
    sample = pd.read_csv(data_dir / 'Sample Submission/BT4012 Competition Submission.csv')
    labels = pd.to_numeric(train.label, errors='coerce')
    known = train.loc[labels.notna()].copy()
    known['label'] = labels.loc[labels.notna()].astype(int)
    assert known.label.isin([0, 1]).all() and len(known) == 31235
    assert len(train) == 141772 and len(test) == len(sample) == 15329
    assert known.txId.is_unique and test['index'].is_unique
    assert sample['index'].equals(test['index'])
    assert np.isfinite(train[FEATURES].to_numpy()).all()
    assert np.isfinite(test[FEATURES].to_numpy()).all()
    return train, known, test, sample, hashes

def fitting_features(learning, cutoff):
    # Repeat the original training-only filter inside each fitting period.
    filter_rows = learning.loc[learning.time_step <= cutoff - 3]
    assert len(filter_rows) > 1 and filter_rows.time_step.max() <= cutoff - 3
    correlations = filter_rows[FEATURES].corrwith(filter_rows.time_step, method='spearman')
    removed = correlations.index[correlations.abs() > .9].tolist()
    return [c for c in FEATURES if c not in removed], removed

def component_prediction(kind, trees, learning, checking, columns, cutoff, end,
                         check_ids, hashes, output, is_final=False):
    settings = XGB_SETTINGS if kind == 'xgb' else LGB_SETTINGS
    receipt = dict(version=VERSION, data_hashes=hashes, kind=kind, trees=trees,
                   settings=settings, columns=columns, cutoff=cutoff, check_end=end,
                   half_life=HALF_LIFE, is_final=is_final,
                   versions=dict(sklearn=sklearn.__version__, xgboost=xgboost.__version__,
                                 lightgbm=lightgbm.__version__, numpy=np.__version__))
    key = hashlib.sha256(json.dumps(receipt, sort_keys=True).encode()).hexdigest()
    cache = output / 'component_predictions'
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f'{key}.npz'
    receipt_path = cache / f'{key}.json'
    if path.exists() and receipt_path.exists():
        assert json.loads(receipt_path.read_text()) == receipt
        with np.load(path, allow_pickle=False) as saved:
            assert np.array_equal(saved['checking_ids'], check_ids)
            assert np.array_equal(saved['learning_ids'], learning.txId.to_numpy())
            assert np.array_equal(saved['learning_labels'], learning.label.to_numpy())
            probabilities = saved['probabilities'].copy()
        return probabilities
    weights = (.5 ** ((learning.time_step.max() - learning.time_step) / HALF_LIFE)).to_numpy()
    weights /= weights.mean()
    model = (XGBClassifier if kind == 'xgb' else LGBMClassifier)(n_estimators=trees, **settings)
    model.fit(learning[columns], learning.label, sample_weight=weights)
    probabilities = model.predict_proba(checking[columns])[:, 1]
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    if is_final:
        models = output / 'models'
        models.mkdir(exist_ok=True)
        if kind == 'xgb':
            model_path = models / f'xgb_{trees}_half4.json'
            model.save_model(model_path)
            reloaded = XGBClassifier()
            reloaded.load_model(model_path)
            reload_probabilities = reloaded.predict_proba(checking[columns])[:, 1]
        else:
            model_path = models / f'lgb_{trees}_half4.txt'
            model.booster_.save_model(str(model_path))
            reloaded = lightgbm.Booster(model_file=str(model_path))
            reload_probabilities = reloaded.predict(checking[columns])
        assert np.allclose(probabilities, reload_probabilities, atol=1e-7, rtol=0)
    temporary = path.with_suffix('.partial.npz')
    np.savez_compressed(temporary, probabilities=probabilities, checking_ids=check_ids,
                        learning_ids=learning.txId.to_numpy(), learning_labels=learning.label.to_numpy())
    temporary.replace(path)
    receipt_path.write_text(json.dumps(receipt, indent=2))
    return probabilities

def run_fixed_comparison(data_dir, output, choice='pending', run=False):
    train, known, test, sample, hashes = load_provided_data(data_dir)
    print(f'Provided rows checked: {len(known):,} confirmed, {len(train)-len(known):,} unknown; {len(test):,} test.')
    print('Reference: 200 XGBoost / 150 LightGBM trees, half-life 4; public AUC 0.95471.')
    print('Student-selected variants: 300/150 and 200/225. No hidden-score forecast.')
    if not run:
        print('PREPARED ONLY: no models fitted, no prediction CSVs generated.')
        return dict(status='prepared_not_trained', rows_confirmed=len(known),
                    rows_unknown=len(train)-len(known), test_rows=len(test), choice=choice)
    if choice not in ('both', 'lgb_only'):
        raise ValueError('Record the student modeling choice before fitting this proposal.')
    chosen = RECIPES.copy()
    if choice == 'lgb_only':
        chosen.pop('long_xgb_300_150')
    output.mkdir(parents=True, exist_ok=True)
    declared = dict(version=VERSION, student_choice=choice, recipes=chosen, folds=FOLDS,
                    xgboost=XGB_SETTINGS, lightgbm=LGB_SETTINGS, half_life=HALF_LIFE,
                    blend_weights=[.5, .5], data_hashes=hashes, automatic_selection=False,
                    external_data=False, kaggle_upload=False,
                    feature_selection='Known fitting rows through cutoff minus three; abs Spearman > .9 removed.')
    (output / 'declared_recipe.json').write_text(json.dumps(declared, indent=2))
    started = time.monotonic()
    rows, per_step, masks = [], [], []
    for cutoff, first, last, group in FOLDS:
        learning = known.loc[known.time_step <= cutoff]
        checking = known.loc[known.time_step.between(first, last)]
        assert learning.time_step.max() < checking.time_step.min()
        xcolumns, removed = fitting_features(learning, cutoff)
        masks.append(dict(cutoff=cutoff, removed=removed, filter_fit_end=cutoff-3))
        components = {}
        for kind, counts, columns in [('xgb', {p[0] for p in chosen.values()}, xcolumns),
                                     ('lgb', {p[1] for p in chosen.values()}, FEATURES)]:
            for trees in sorted(counts):
                components[(kind, trees)] = component_prediction(
                    kind, trees, learning, checking, columns, cutoff, last,
                    checking.txId.to_numpy(), hashes, output)
        for name, (xtrees, ltrees) in chosen.items():
            p = .5*components[('xgb', xtrees)] + .5*components[('lgb', ltrees)]
            rows.append(dict(cutoff=cutoff, check_first=first, check_last=last, group=group,
                             recipe=name, training_rows=len(learning), checking_rows=len(checking),
                             auc=float(roc_auc_score(checking.label, p))))
            for step in checking.time_step.unique():
                selected = checking.time_step.to_numpy() == step
                y = checking.label.to_numpy()[selected]
                per_step.append(dict(cutoff=cutoff, step=int(step), recipe=name,
                                     rows=int(selected.sum()), positives=int(y.sum()),
                                     auc=float(roc_auc_score(y, p[selected])) if len(np.unique(y)) == 2 else None))
        pd.DataFrame(rows).to_csv(output / 'validation_results.csv', index=False)
        (output / 'progress.json').write_text(json.dumps(dict(phase='validation', last_cutoff=cutoff,
                                      completed_checks=len(masks), elapsed_seconds=time.monotonic()-started)))
        print(f'Completed training through {cutoff}, checking {first}-{last}.', flush=True)
    scores = pd.DataFrame(rows)
    summaries = []
    views = {'original_15': scores.group.eq('original_14_period'),
             'original_later_seven': scores.cutoff.between(15, 21),
             'later_cutoffs_24_28_31': scores.group.eq('later_cutoff')}
    for view, selected in views.items():
        frame = scores.loc[selected]
        control = frame.loc[frame.recipe.eq('reference_200_150')].set_index('cutoff').auc
        for name in chosen:
            values = frame.loc[frame.recipe.eq(name)].set_index('cutoff').auc
            delta = values-control
            summaries.append(dict(group=view, recipe=name, checks=len(values), mean_auc=values.mean(),
                                  worst_auc=values.min(), mean_delta_vs_reference=delta.mean(),
                                  wins_vs_reference=int((delta > 0).sum())))
    pd.DataFrame(summaries).to_csv(output / 'summary.csv', index=False)
    pd.DataFrame(per_step).to_csv(output / 'per_step_auc.csv', index=False)
    (output / 'training_only_feature_masks.json').write_text(json.dumps(masks, indent=2))
    xcolumns, removed = fitting_features(known, 35)
    assert set(removed) == {'feat_100','feat_101','feat_103','feat_136','feat_137','feat_139'}
    components = {}
    for kind, counts, columns in [('xgb', {p[0] for p in chosen.values()}, xcolumns),
                                 ('lgb', {p[1] for p in chosen.values()}, FEATURES)]:
        for trees in sorted(counts):
            components[(kind, trees)] = component_prediction(
                kind, trees, known, test, columns, 35, 49, test['index'].to_numpy(), hashes, output, True)
    submission_hashes = {}
    for name, (xtrees, ltrees) in chosen.items():
        submission = sample[['index']].copy()
        p = .5*components[('xgb', xtrees)] + .5*components[('lgb', ltrees)]
        submission['target'] = p
        assert submission['index'].equals(sample['index']) and submission.target.between(0, 1).all()
        path = output / f'submission_half4_{name}_experimental.csv'
        submission.to_csv(path, index=False)
        checked = pd.read_csv(path)
        assert checked['index'].equals(sample['index']) and np.allclose(checked.target, p, atol=1e-15, rtol=0)
        submission_hashes[name] = digest(path)
        if name == 'reference_200_150':
            old_path = data_dir.parent / 'claude_experiments/submissions/submission_lowreg_pair_recent4.csv'
            if old_path.exists():
                old = pd.read_csv(old_path)
                assert old['index'].equals(submission['index'])
                assert np.max(np.abs(old.target-p)) <= 1e-7
    assert {name: digest(data_dir/name) for name in EXPECTED_HASHES} == hashes
    (output / 'progress.json').write_text(json.dumps(dict(phase='complete', checks=len(FOLDS),
                              elapsed_seconds=time.monotonic()-started, submission_hashes=submission_hashes,
                              final_selection='student decision', kaggle_upload=False), indent=2))
    print(pd.DataFrame(summaries).to_string(index=False))
    print('Fixed comparison complete; review results before selecting a submission.')
    return summaries

def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-dir', type=Path, default=Path.cwd()/'Data Set')
    parser.add_argument('--output-dir', type=Path, default=Path.cwd()/'half4_longer_training_results')
    parser.add_argument('--choice', choices=['pending', 'both', 'lgb_only'], default='pending')
    parser.add_argument('--run', action='store_true')
    args = parser.parse_args()
    return run_fixed_comparison(args.data_dir, args.output_dir, args.choice, args.run)



In [3]:
result = run_fixed_comparison(DATA_DIR, OUTPUT_DIR, choice=STUDENT_CHOICE, run=RUN_TRAINING)


Provided rows checked: 31,235 confirmed, 110,537 unknown; 15,329 test.
Reference: 200 XGBoost / 150 LightGBM trees, half-life 4; public AUC 0.95471.
Student-selected variants: 300/150 and 200/225. No hidden-score forecast.
Completed training through 7, checking 8-21.
Completed training through 8, checking 9-22.
Completed training through 9, checking 10-23.
Completed training through 10, checking 11-24.
Completed training through 11, checking 12-25.
Completed training through 12, checking 13-26.
Completed training through 13, checking 14-27.
Completed training through 14, checking 15-28.
Completed training through 15, checking 16-29.
Completed training through 16, checking 17-30.
Completed training through 17, checking 18-31.
Completed training through 18, checking 19-32.
Completed training through 19, checking 20-33.
Completed training through 20, checking 21-34.
Completed training through 21, checking 22-35.
Completed training through 24, checking 25-35.
Completed training through 28,

## Reproduction and assistance record

The code generates all three fixed recipes and checks the original half-life-four reference when its CSV is present. Fitting needs only the supplied train/test/sample files; that optional reference file is not needed to produce predictions. Adjust DATA_DIR and OUTPUT_DIR for another environment.

OpenAI Codex reviewed prior results, suggested and explained this fixed comparison, and implemented it after the student's explicit modeling choice. The student remains responsible for interpretation, future model choices, any submission and accurate AI disclosure. Source data, earlier experiment files and previously submitted CSVs are preserved.

### 43.1. Student-selected longer training and measured result

**13 October 2026.** After the shorter-training comparison, the student chose **“Test both longer-training variants (Recommended)”**. The fixed comparison retains the half-life-four reference (200 XGBoost / 150 LightGBM trees), adds 300/150 and adds 200/225. Each alternative changes one component's tree count. Half-life four, the 50:50 probability average, learning rate 0.05, 15 leaves, bin settings, minimum-leaf settings, lambda 1, seed 42 and two CPU threads remain unchanged.

The run completed the same **18 chronological checks** and full fitting in **46.3 seconds** on CPU. It reused **38 verified reference components** and fitted **38 new components**. Reuse required identical data hashes, parameters, package versions, training-only feature lists, fitting labels and checking IDs. The fitting function is unchanged; independent verification confirmed the reused receipts, probability arrays, models, checking scores and reference CSV are identical to the previous completed comparison. No earlier experiment was overwritten.

Final fitting uses all **31,235 confirmed labels** through step 35. The **110,537 unknown rows** receive no invented labels and are excluded from supervised fitting. Each recipe generates **15,329 aligned test predictions**. No automatic parameter search, extra mixture, winner selection or Kaggle upload occurred.

| Check group | Reference 200/150 | Longer XGBoost 300/150 | Longer LightGBM 200/225 |
|---|---:|---:|---:|
| Original 15 | 0.950128 | 0.950617 | 0.948387 |
| Original later seven | 0.915564 | 0.916658 | 0.911743 |
| Three later cutoffs | 0.997337 | 0.997439 | 0.997403 |

### Longer XGBoost: small gains with tradeoffs

The 300/150 recipe raises the original-15 mean by **+0.000489 AUC**, winning **9/15**. It raises the original later-seven mean by **+0.001094**, winning **4/7**. It also wins **3/3** added later-cutoff checks, with a mean gain of only **+0.000102**.

The latest fourteen-period check (fit through 21, check 22–35) declines by **-0.000248**. The worst original-check AUC declines by **-0.000256**. This is not a uniform improvement. The 300-tree component makes additional corrections using the same recency weights, which changes its predictions and its contribution to the probability average. No unique cause of the mixed gains is proved by the aggregate scores.

The candidate's test-rank correlation with the public reference is **0.997916**, sharing **976 of the top 1,000** predicted transactions. These are label-free similarity measures, not accuracy estimates or a confidence interval.

### Longer LightGBM: weaker main means

The 200/225 recipe changes the original-15 mean by **-0.001741**, winning **6/15**. Its later-seven mean changes by **-0.003820**, winning only **1/7**. It wins the three added later-cutoff checks by tiny margins, but that does not remove the deterioration on the main fourteen-period checks. Its latest fourteen-period and worst-check scores also decline.

**AI-assisted assessment:** The 300-XGBoost/150-LightGBM recipe has the stronger evidence among these two new candidates. It raises the original-15 and later-seven means and wins all three added later-cutoff checks, but its latest fourteen-period and worst-check scores fall slightly. The gains are small and mixed, and the checks overlap prior development. It is a reasonable experimental candidate for student review, not a demonstrated replacement for the 0.95471 public reference or a calibrated prediction above 0.96. The 225-tree LightGBM alternative has weaker main means. Do not automatically select a final model or combine both new settings.

The currently verified submitted public best remains **0.95471**, **0.00529 below** 0.96000. No public or private score is known for either new candidate. The observed local AUC gains cannot be added to 0.95471 to forecast a public score. A decision to submit the 300/150 candidate would test that exact recipe on the public subset; it would not establish private performance or prove that longer XGBoost training is generally better.

### Later endpoints and validation limits

| Train through | Check steps | Reference 200/150 | Longer XGBoost 300/150 | Longer LightGBM 200/225 |
|---:|---|---:|---:|---:|
| 21 | 22–35 | 0.903163 | 0.902915 | 0.901954 |
| 24 | 25–35 | 0.997268 | 0.997317 | 0.997312 |
| 28 | 29–35 | 0.996713 | 0.996929 | 0.996823 |
| 31 | 32–35 | 0.998030 | 0.998072 | 0.998074 |

The added later checks forecast 11, 7 and 4 periods. Their approximately 0.997 means cannot be treated as a fourteen-period Kaggle forecast. All checking periods overlap earlier development data, and their labels have been repeatedly examined. Across the 18 checks, **181,071 checking appearances** cover **21,682 unique transactions**.

Each check learns XGBoost's time-feature filter using only confirmed fitting rows through cutoff minus three, removing features with absolute time Spearman correlation above 0.9. Reference and candidates share the same cutoff-specific mask. This retains the repaired validation protocol from Section 42. Compare these scores against the same nested-filter control; older hardcoded-mask averages are different development evaluations. Fitting/checking IDs are disjoint and no checking row fits a model or filter.

Final fitting produces the same six XGBoost exclusions as the public reference: feat_100, feat_101, feat_103, feat_136, feat_137 and feat_139. LightGBM retains all 165 features. Test features are prediction inputs only.

### Verification and reproducibility

An independent process recalculated every AUC, per-period score, group mean, worst score, delta and win count from saved component probabilities. It checked receipt hashes, fixed settings, fitting/checking IDs and labels, nested feature masks, reused references and every probability formula. All four final models were reloaded in a fresh process and reproduced their saved probabilities; actual tree counts match 200/300 for XGBoost and 150/225 for LightGBM. Every CSV passes row count, exact sample/test order, unique-index, finite-probability, range and round-trip checks. Original source data hashes and the original submitted file are unchanged. The reproduced reference differs from the stored public CSV by at most **5.9604645e-08**.

New files:

- submission_half4_long_xgb_300_150_experimental.csv — SHA-256 **7fd1ccf24d89a1ff1be9d21a273f787e1a40f6413d98a324fd92ba8767d95163**.
- submission_half4_long_lgb_200_225_experimental.csv — SHA-256 **e709f3c34a63390b7f9d9376d41893d72398566c1cbfd399e545f05df4c53cd8**.

The standalone executed notebook is e1156285_half4_longer_training.ipynb. The exact source, declared settings, cache-reuse evidence, checking scores, four full models, final CSVs and independent verification are in half4_longer_training_results/. To reproduce in a fresh folder, adjust DATA_DIR and OUTPUT_DIR; the code fits all required components from the supplied train/test/sample files. Optional stored references are used for verification, never required to generate predictions.

**AI assistance record:** OpenAI Codex proposed and explained this fixed comparison after the shorter-training result. The student selected both longer-training variants and the stated checking protocol. Codex executed only that selection, reused verified identical controls, fitted the new components, independently verified the artifacts and drafted this factual review. The student remains responsible for interpretation, subsequent modeling choices, any submission and disclosure. No external transaction data or labels, AutoML, automatic final selection or Kaggle upload was used.
